# Imports

In [2]:
# pip install -e D:\Python_Project2\Hypothesis_generation_agent-refined-prompts-ontho-expansion\HypothesisGeneration

In [3]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModel
import networkx as nx
from pathlib import Path

In [4]:
from HypothesisGeneration import (
    make_dir_if_needed,
    load_graph_with_text_as_JSON,
    return_giant_component_of_graph,
    load_embeddings,
    generate_node_embeddings,
    SciAgentsGraphFlow,
)
from GraphReasoning import (
    save_embeddings,
)

In [5]:
data_dir_source = Path("../inputs/initial_graph_biomed")
data_dir_output = Path("../outputs/initial_ouput_biomed")
make_dir_if_needed(data_dir_output)
make_dir_if_needed(data_dir_source)


'Directory already exists.'

## Tokenizer and Embedding

In [7]:
tokenizer_model = "BAAI/bge-large-en-v1.5"
embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
embedding_model = AutoModel.from_pretrained(
    tokenizer_model,
)

## Graph

In [9]:
graph_name = "sparse_global_graph.graphml"  # Second example in notebook
G = load_graph_with_text_as_JSON(data_dir=data_dir_source, graph_name=graph_name)
G = return_giant_component_of_graph(G)  # S
G = nx.Graph(G)

100%|████████████████████████████████████████████████████████████████████████| 37951/37951 [00:00<00:00, 111493.10it/s]


## Embeddings


In [11]:
embedding_name = "embeddings-sparse_global_graph-bge_large_en_v1.5.pkl"
filename = f"{embedding_name}"
try:
    node_embeddings = load_embeddings(f"{data_dir_source}/{embedding_name}")
except:
    print("Node embeddings not loaded, need to regenerate.")
    node_embeddings = generate_node_embeddings(
        G,
        embedding_tokenizer,
        embedding_model,
    )
    print("Node embeddings regenerated. Saving them to avoid recomputation")
    save_embeddings(node_embeddings, f"{data_dir_source}/{embedding_name}")


##  Default LLM model per agent

- Path Generation Agent: gpt-5-nano
- Ontologist: gpt-5-mini
- Scientist: gpt-5 (reasoning)
- Hypothesis Expansion Agent: gpt-5-mini
- Novelty Critic Agent: gpt-5-mini
- Coding Tasks Agent: gpt-5

- Summary Agent: gpt-5-mini

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = ""
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = ""

In [15]:
from openai import OpenAI
import os

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

sci_agents_group = SciAgentsGraphFlow(
    G=G,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    node_embeddings=node_embeddings,
    llm_client=openai_client, 
    use_llm_validation=True,
)

TypeError: SciAgentsGraphFlow.__init__() got an unexpected keyword argument 'llm_client'

In [ ]:
import asyncio

await sci_agents_group.run_with_console(
    message="Discover research idea between tumor lysis syndrome (TLS)||Disease and gene"
    ,
    save_to=data_dir_output / "example_run.txt",
)